# E1 · W6 · Kaggle run 15 — video demo có vẽ ROI và cảnh báo (CPU)
**Mục đích (issue #76, ADR-0009 Accepted):** dựng 3 video demo từ bằng chứng run 14. Mỗi video có vẽ:
- polygon ROI `center_hex`;
- hộp nhãn gốc;
- track có ID;
- dải cảnh báo R1 (theo track) và R0 (detector-only);
- dấu cảnh báo sớm.

Cảnh báo được tính lại bằng `alert_engine.py` với đúng hai file cấu hình đã chốt (job kiểm SHA-256). Video được chọn theo luật ghi trong job:
1. video Drone có cảnh báo R1 `vao_vung` dài nhất;
2. video không phải Drone có cảnh báo sai R0 dài nhất;
3. video Drone có nhiều ID switch nhất.

Không cần GPU.

**Trước khi chạy:**
1. Dataset `e1-w3-tools` → **New Version**, thêm từ commit `COMMIT`:
   - `scripts/kaggle/w6_alert_demo.py`;
   - `scripts/alert/alert_engine.py`;
   - `configs/roi_w6.yaml`, `configs/alert_e1.yaml`.
   
   Giữ `w3_export_yolo.py`.
2. *Add Input*:
   - `e1-svanstrom-raw`;
   - `e1-w3-tools` (bản mới);
   - output notebook `e1-w6-run14`.
3. *Settings*: Accelerator **None** (CPU), Internet **On**. Điền `COMMIT`.

**Sau khi chạy:**
- `w6_alert_demo_evidence.zip` → thư mục kết quả run 15;
- `w6_alert_demo_videos.zip` (mp4) → Drive `demo/W06`, **không** đưa vào Git.


In [ ]:
COMMIT = ""                      # <-- SHA day du 40 ky tu
ROI_SHA = "b8babbbcd19cb702190d970d16e6073bfc7bc020a0fde4270983db84ca2c6337"      # configs/roi_w6.yaml (ADR-0009), SHA ban LF
ALERT_SHA = "2d4cee9f0cc7e3bd63eb616dc426f2a149aedf3e49d8504df9c32f4eeff8d2c9"    # configs/alert_e1.yaml (ADR-0009)


In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install mcos-decoder
import cv2, yaml
print("cv2", cv2.__version__)


In [ ]:
# Tim thu muc du lieu tho; neu dataset tho la file .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if "e1-yolo" in z:
            continue
        print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
SCRIPT = find("w6_alert_demo.py")
for n in ["alert_engine.py", "w3_export_yolo.py"]:
    assert os.path.dirname(find(n)) == os.path.dirname(SCRIPT), f"{n} phai nam cung thu muc voi w6_alert_demo.py"
ROI, ALERT = find("roi_w6.yaml"), find("alert_e1.yaml")
def sha_lf(p):                       # bo qua khac biet CRLF/LF khi file duoc lay tu Windows
    return hashlib.sha256(open(p, "rb").read().replace(b"\r\n", b"\n")).hexdigest()
assert sha_lf(ROI) == ROI_SHA and sha_lf(ALERT) == ALERT_SHA, "file cau hinh khac ban da chot trong ADR-0009 - kiem tra version e1-w3-tools"
EVID = os.path.dirname(find("w6_run14/stage_status.json"))
for p in (SCRIPT, ROI, ALERT, EVID + "/inputs.json"):
    print(f"sha256={sha(p)[:16]}  {p}")
print(json.load(open(EVID + "/inputs.json"))["weights_sha256"][:16], "(phai bat dau a3de1e01)")


In [ ]:
!{sys.executable} {SCRIPT} --data-root {DATA_ROOT} --evidence {EVID} --roi {ROI} --alert {ALERT} --out /kaggle/working --commit {COMMIT}


## Tự kiểm tra (phải in `PASS`)

In [ ]:
out = "/kaggle/working/w6_alert_demo"
st = json.load(open(f"{out}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
for d in json.load(open(f"{out}/demo_index.json")):
    print(d["video_id"], "|", d["rule"], "|", d["frames"], "frame")
ok = all(v["status"] == "OK" for v in st.values()) and len(st) == 5
print("PASS" if ok else "FAIL - gui stage_status.json cho Claude")
for z in ("w6_alert_demo_evidence.zip", "w6_alert_demo_videos.zip"):
    p = f"/kaggle/working/{z}"; print(z, os.path.getsize(p) // 1024, "KB  sha256", sha(p))
